In [1]:
import pandas as pd
import os

In [2]:
sentences_file = "data/frases-musicas-final-filled.csv"
annotations_dir = "data/annotations/20251016"
assignments_file = "data/assignments.csv"

In [3]:
sentences_df = pd.read_csv(sentences_file)
assignments_df = pd.read_csv(assignments_file)

In [4]:
sentences_df.head()

,music_id,music_frase_id,frase,frase_id,tem_padrao,padrao_encontrado
0,1,1,Carolina é uma menina bem difícil de esquecer.,1,Female,"[(1916877163388338700, 3, 6), (191687716338833..."
1,1,2,Andar bonito e um brilho no olhar.,2,N,NaN
2,1,3,Tem um jeito adolescente que me faz enlouquecer.,3,N,NaN
3,1,4,E um molejo que eu não vou te enganar.,4,N,NaN
4,1,5,"Maravilha feminina, meu docinho de pavê.",5,N,NaN


In [5]:
assignments_df.head()

,annotator_id,frase_id
0,user01,394315
1,user01,650570
2,user01,119894
3,user01,422159
4,user01,2863507


In [6]:
# lista todos os arquivos do diretorio de anotações
annotation_files = [
    os.path.join(annotations_dir, f)
    for f in os.listdir(annotations_dir)
    if f.endswith(".csv") and f.startswith("annotation_user")
]
annotation_files

['data/annotations/20251016/annotation_user34.csv',
 'data/annotations/20251016/annotation_user20.csv',
 'data/annotations/20251016/annotation_user21.csv',
 'data/annotations/20251016/annotation_user35.csv',
 'data/annotations/20251016/annotation_user09.csv',
 'data/annotations/20251016/annotation_user23.csv',
 'data/annotations/20251016/annotation_user22.csv',
 'data/annotations/20251016/annotation_user26.csv',
 'data/annotations/20251016/annotation_user32.csv',
 'data/annotations/20251016/annotation_user33.csv',
 'data/annotations/20251016/annotation_user27.csv',
 'data/annotations/20251016/annotation_user31.csv',
 'data/annotations/20251016/annotation_user25.csv',
 'data/annotations/20251016/annotation_user18.csv',
 'data/annotations/20251016/annotation_user24.csv',
 'data/annotations/20251016/annotation_user30.csv',
 'data/annotations/20251016/annotation_user29.csv',
 'data/annotations/20251016/annotation_user15.csv',
 'data/annotations/20251016/annotation_user01.csv',
 'data/annot

In [7]:
annotations_df = None
if len(annotation_files) > 0:
    annotations_df = pd.read_csv(annotation_files[0])
    for f in annotation_files[1:]:
        df = pd.read_csv(f)
        annotations_df = pd.concat([annotations_df, df], ignore_index=True)
annotations_df

,user,frase_id,annotation
0,user34,2264732,nao
1,user34,3344723,sim
2,user34,2759746,sim
3,user34,3816656,sim
4,user34,2068794,nao
...,...,...,...
46228,user11,4548088,sim
46229,user11,3160527,sim
46230,user11,2710599,sim
46231,user11,49418,nao_sei


In [8]:
assignments_with_annotations = assignments_df.merge(
    annotations_df,
    left_on=["annotator_id", "frase_id"],
    right_on=["user", "frase_id"],
    how="left",
    
)
assignments_with_annotations

,annotator_id,frase_id,user,annotation
0,user01,394315,user01,sim
1,user01,650570,user01,nao
2,user01,119894,user01,sim
3,user01,422159,user01,nao_sei
4,user01,2863507,user01,nao
...,...,...,...,...
69995,user35,2324000,NaN,NaN
69996,user35,2227938,NaN,NaN
69997,user35,1217206,NaN,NaN
69998,user35,376277,NaN,NaN


In [9]:
# em assignments_with_annotations, caso user seja NaN, preencher com o valor de annotator_id
assignments_with_annotations["user"] = assignments_with_annotations["user"].fillna(assignments_with_annotations["annotator_id"])
assignments_with_annotations["annotation"] = assignments_with_annotations["annotation"].fillna("not_annotated") 
assignments_with_annotations

,annotator_id,frase_id,user,annotation
0,user01,394315,user01,sim
1,user01,650570,user01,nao
2,user01,119894,user01,sim
3,user01,422159,user01,nao_sei
4,user01,2863507,user01,nao
...,...,...,...,...
69995,user35,2324000,user35,not_annotated
69996,user35,2227938,user35,not_annotated
69997,user35,1217206,user35,not_annotated
69998,user35,376277,user35,not_annotated


In [10]:
assignments_with_annotations = assignments_with_annotations.merge(
    sentences_df[["frase_id", "frase", "tem_padrao"]],
    on="frase_id",
    how="left"
)
assignments_with_annotations

,annotator_id,frase_id,user,annotation,frase,tem_padrao
0,user01,394315,user01,sim,Hoje sou um homem casado.,Male
1,user01,650570,user01,nao,Mas lendo atingi o bom senso.,N
2,user01,119894,user01,sim,Todas as mulheres estão atentas.,Female
3,user01,422159,user01,nao_sei,"Homem que tem mulher feia, meu.",Female
4,user01,2863507,user01,nao,garota igual.,Female
...,...,...,...,...,...,...
69995,user35,2324000,user35,not_annotated,É homem forte.,Male
69996,user35,2227938,user35,not_annotated,Que ele é tão grande.,Male
69997,user35,1217206,user35,not_annotated,"E ai poderemos sorrir como mulheres negras,.",Female
69998,user35,376277,user35,not_annotated,Ela estava tão linda na quela janela.,Female


In [ ]:
awa = assignments_with_annotations.copy()

awa["frase_id"] = awa["frase_id"].astype("category")
awa["annotation"] = pd.Categorical(
    awa["annotation"],
    categories=["sim", "nao", "nao_sei", "not_annotated"]
)

annotation_statistics_df = (
    awa.groupby(["frase_id", "annotation"]).size()
      .unstack(fill_value=0)              
      .reindex(columns=["sim", "nao", "nao_sei", "not_annotated"], fill_value=0)
)

annotation_statistics_df["total_annotations"] = annotation_statistics_df.sum(axis=1)
annotation_statistics_df["valid_annotations"] = annotation_statistics_df[["sim", "nao", "nao_sei"]].sum(axis=1)

annotation_statistics_df

/var/folders/4g/hm9d3pn57nv1p17qlv8k47mc0000gn/T/ipykernel_3064/3052650644.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  awa.groupby(["frase_id", "annotation"]).size()


annotation,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations
frase_id,,,,,,
1,1,0,0,3,4,1
6,2,0,0,2,4,2
51,2,0,0,1,3,2
798,3,0,0,0,3,3
1235,1,0,0,1,2,1
...,...,...,...,...,...,...
4572971,1,0,1,2,4,2
4572997,2,2,0,0,4,4
4573057,1,0,0,2,3,1


In [39]:
annotation_statistics_df[annotation_statistics_df['valid_annotations'] > 0]

annotation,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations
frase_id,,,,,,
1,1,0,0,3,4,1
6,2,0,0,2,4,2
51,2,0,0,1,3,2
798,3,0,0,0,3,3
1235,1,0,0,1,2,1
...,...,...,...,...,...,...
4572971,1,0,1,2,4,2
4572997,2,2,0,0,4,4
4573057,1,0,0,2,3,1


In [41]:
annotation_statistics_df[annotation_statistics_df['valid_annotations'] >= 2]

annotation,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations
frase_id,,,,,,
6,2,0,0,2,4,2
51,2,0,0,1,3,2
798,3,0,0,0,3,3
1277,2,0,0,1,3,2
1280,2,0,0,1,3,2
...,...,...,...,...,...,...
4572890,1,3,0,0,4,4
4572971,1,0,1,2,4,2
4572997,2,2,0,0,4,4


In [12]:
annotation_statistics_df['valid_annotations'].describe()

count    18831.000000
mean         2.455154
std          1.741955
min          0.000000
25%          2.000000
50%          2.000000
75%          3.000000
max         26.000000
Name: valid_annotations, dtype: float64

In [13]:
annotation_statistics_df['valid_annotations'].value_counts()

valid_annotations
2     7897
3     5835
1     3051
4     1895
0       53
22      31
23      26
24      26
25       8
21       7
26       2
Name: count, dtype: int64

In [14]:
annotation_statistics_df_full = pd.merge(
    annotation_statistics_df.reset_index(),
    sentences_df[["frase_id", "frase", "tem_padrao"]],
    on="frase_id",
    how="left"
)
annotation_statistics_df_full

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao
0,1,1,0,0,3,4,1,Carolina é uma menina bem difícil de esquecer.,Female
1,6,2,0,0,2,4,2,"Inteligente, ela é muito sensual.",Female
2,51,2,0,0,1,3,2,"Menina bela, menina bela.",Female
3,798,3,0,0,0,3,3,Ele não é feliz.,Male
4,1235,1,0,0,1,2,1,"Ela é linda, mas não tem nome.",Female
...,...,...,...,...,...,...,...,...,...
18826,4572971,1,0,1,2,4,2,Sem que a matéria esteja sob a influência do s...,Female
18827,4572997,2,2,0,0,4,4,"A doença é menos que um extrato de pó de nada,...",Female
18828,4573057,1,0,0,2,3,1,O homem mentalmente espiritualizado é falante.,Male
18829,4573086,3,0,1,0,4,4,"O homem atual, na grande maioria, é carente da...",Male


In [15]:
annotation_statistics_df_full['tem_padrao'].value_counts()

tem_padrao
Female    11314
Male       7417
N           100
Name: count, dtype: int64

In [16]:
annotation_statistics_df_full["real_class"] = annotation_statistics_df_full["tem_padrao"].map({
    "Female": "sim",
    "Male": "sim",
    "N": "nao"
})
annotation_statistics_df_full

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class
0,1,1,0,0,3,4,1,Carolina é uma menina bem difícil de esquecer.,Female,sim
1,6,2,0,0,2,4,2,"Inteligente, ela é muito sensual.",Female,sim
2,51,2,0,0,1,3,2,"Menina bela, menina bela.",Female,sim
3,798,3,0,0,0,3,3,Ele não é feliz.,Male,sim
4,1235,1,0,0,1,2,1,"Ela é linda, mas não tem nome.",Female,sim
...,...,...,...,...,...,...,...,...,...,...
18826,4572971,1,0,1,2,4,2,Sem que a matéria esteja sob a influência do s...,Female,sim
18827,4572997,2,2,0,0,4,4,"A doença é menos que um extrato de pó de nada,...",Female,sim
18828,4573057,1,0,0,2,3,1,O homem mentalmente espiritualizado é falante.,Male,sim
18829,4573086,3,0,1,0,4,4,"O homem atual, na grande maioria, é carente da...",Male,sim


In [17]:
annotation_statistics_df_full['majority_annotation'] = annotation_statistics_df_full[["sim", "nao", "nao_sei"]].idxmax(axis=1)
annotation_statistics_df_full

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class,majority_annotation
0,1,1,0,0,3,4,1,Carolina é uma menina bem difícil de esquecer.,Female,sim,sim
1,6,2,0,0,2,4,2,"Inteligente, ela é muito sensual.",Female,sim,sim
2,51,2,0,0,1,3,2,"Menina bela, menina bela.",Female,sim,sim
3,798,3,0,0,0,3,3,Ele não é feliz.,Male,sim,sim
4,1235,1,0,0,1,2,1,"Ela é linda, mas não tem nome.",Female,sim,sim
...,...,...,...,...,...,...,...,...,...,...,...
18826,4572971,1,0,1,2,4,2,Sem que a matéria esteja sob a influência do s...,Female,sim,sim
18827,4572997,2,2,0,0,4,4,"A doença é menos que um extrato de pó de nada,...",Female,sim,sim
18828,4573057,1,0,0,2,3,1,O homem mentalmente espiritualizado é falante.,Male,sim,sim
18829,4573086,3,0,1,0,4,4,"O homem atual, na grande maioria, é carente da...",Male,sim,sim


In [18]:
annotation_statistics_df_full['agreement_rate'] = annotation_statistics_df_full[["sim", "nao", "nao_sei"]].max(axis=1) * 100 / annotation_statistics_df_full['valid_annotations']
annotation_statistics_df_full

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class,majority_annotation,agreement_rate
0,1,1,0,0,3,4,1,Carolina é uma menina bem difícil de esquecer.,Female,sim,sim,100.0
1,6,2,0,0,2,4,2,"Inteligente, ela é muito sensual.",Female,sim,sim,100.0
2,51,2,0,0,1,3,2,"Menina bela, menina bela.",Female,sim,sim,100.0
3,798,3,0,0,0,3,3,Ele não é feliz.,Male,sim,sim,100.0
4,1235,1,0,0,1,2,1,"Ela é linda, mas não tem nome.",Female,sim,sim,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
18826,4572971,1,0,1,2,4,2,Sem que a matéria esteja sob a influência do s...,Female,sim,sim,50.0
18827,4572997,2,2,0,0,4,4,"A doença é menos que um extrato de pó de nada,...",Female,sim,sim,50.0
18828,4573057,1,0,0,2,3,1,O homem mentalmente espiritualizado é falante.,Male,sim,sim,100.0
18829,4573086,3,0,1,0,4,4,"O homem atual, na grande maioria, é carente da...",Male,sim,sim,75.0


In [19]:
annotation_statistics_df_full_final = annotation_statistics_df_full.copy()
annotation_statistics_df_full_final = annotation_statistics_df_full_final[annotation_statistics_df_full_final['majority_annotation'] != 'nao_sei']
annotation_statistics_df_full_final

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class,majority_annotation,agreement_rate
0,1,1,0,0,3,4,1,Carolina é uma menina bem difícil de esquecer.,Female,sim,sim,100.0
1,6,2,0,0,2,4,2,"Inteligente, ela é muito sensual.",Female,sim,sim,100.0
2,51,2,0,0,1,3,2,"Menina bela, menina bela.",Female,sim,sim,100.0
3,798,3,0,0,0,3,3,Ele não é feliz.,Male,sim,sim,100.0
4,1235,1,0,0,1,2,1,"Ela é linda, mas não tem nome.",Female,sim,sim,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
18826,4572971,1,0,1,2,4,2,Sem que a matéria esteja sob a influência do s...,Female,sim,sim,50.0
18827,4572997,2,2,0,0,4,4,"A doença é menos que um extrato de pó de nada,...",Female,sim,sim,50.0
18828,4573057,1,0,0,2,3,1,O homem mentalmente espiritualizado é falante.,Male,sim,sim,100.0
18829,4573086,3,0,1,0,4,4,"O homem atual, na grande maioria, é carente da...",Male,sim,sim,75.0


In [20]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(annotation_statistics_df_full_final['real_class'], annotation_statistics_df_full_final['majority_annotation'], labels=['sim', 'nao'])
cm

array([[16310,  2264],
       [    8,    92]])

In [51]:
annotation_statistics_df_full_final['agreement_rate'].describe()

count    18621.000000
mean        88.831286
std         19.031601
min         33.333333
25%         75.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: agreement_rate, dtype: float64

In [21]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

y_true = annotation_statistics_df_full_final['real_class']
y_pred = annotation_statistics_df_full_final['majority_annotation']

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"Acurácia:  {acc:.4f}")
print(f"Precisão:  {prec:.4f}")
print(f"Revocação: {rec:.4f}")
print(f"F1-score:  {f1:.4f}")



Acurácia:  0.8783
Precisão:  0.9944
Revocação: 0.8783
F1-score:  0.9303


In [22]:
print(classification_report(y_true, y_pred, zero_division=0))

              precision    recall  f1-score   support

         nao       0.04      0.92      0.07       100
         sim       1.00      0.88      0.93     18574

    accuracy                           0.88     18674
   macro avg       0.52      0.90      0.50     18674
weighted avg       0.99      0.88      0.93     18674



In [25]:
classes_no = annotation_statistics_df_full_final.copy()[annotation_statistics_df_full_final['majority_annotation'] == 'nao']
classes_no

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class,majority_annotation,agreement_rate
28,7235,0,3,0,1,4,3,"Fuma um vape com gosto de ""hoje a mãe arrasa"" ...",Female,sim,nao,100.000000
43,9994,0,2,0,2,4,2,Se o senhor não tá lembrado.,Male,sim,nao,100.000000
59,14630,0,1,0,2,3,1,Todos eles afim de entregar os irmão.,Male,sim,nao,100.000000
60,14647,0,1,0,2,3,1,De um ônibus pra outro aquilo para ele era o fim.,Male,sim,nao,100.000000
85,17793,0,2,0,0,2,2,Porque a confiança é uma mulher ingrata.,Female,sim,nao,100.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
18803,4570171,0,2,0,2,4,2,Da parte mais alta de um templo ele foi lançado.,Male,sim,nao,100.000000
18806,4570197,0,2,0,1,3,2,Nero ele foi decapitado\n.,Male,sim,nao,100.000000
18818,4571613,1,2,0,0,3,3,O Mestre ensinando que todos os homens são igu...,Male,sim,nao,66.666667
18824,4572851,0,1,0,2,3,1,Embora a inteligência do homem seja grande par...,Male,sim,nao,100.000000


In [27]:
classes_no['agreement_rate'].describe()

count    2356.000000
mean       85.485706
std        18.148336
min        50.000000
25%        66.666667
50%       100.000000
75%       100.000000
max       100.000000
Name: agreement_rate, dtype: float64

In [43]:
classes_no_final = classes_no[(classes_no['agreement_rate'] >= 60) & (classes_no['total_annotations'] >= 2)]
classes_no_final

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class,majority_annotation,agreement_rate
28,7235,0,3,0,1,4,3,"Fuma um vape com gosto de ""hoje a mãe arrasa"" ...",Female,sim,nao,100.000000
43,9994,0,2,0,2,4,2,Se o senhor não tá lembrado.,Male,sim,nao,100.000000
59,14630,0,1,0,2,3,1,Todos eles afim de entregar os irmão.,Male,sim,nao,100.000000
60,14647,0,1,0,2,3,1,De um ônibus pra outro aquilo para ele era o fim.,Male,sim,nao,100.000000
85,17793,0,2,0,0,2,2,Porque a confiança é uma mulher ingrata.,Female,sim,nao,100.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
18803,4570171,0,2,0,2,4,2,Da parte mais alta de um templo ele foi lançado.,Male,sim,nao,100.000000
18806,4570197,0,2,0,1,3,2,Nero ele foi decapitado\n.,Male,sim,nao,100.000000
18818,4571613,1,2,0,0,3,3,O Mestre ensinando que todos os homens são igu...,Male,sim,nao,66.666667
18824,4572851,0,1,0,2,3,1,Embora a inteligência do homem seja grande par...,Male,sim,nao,100.000000


In [49]:
classes_no_final['tem_padrao'].value_counts()

tem_padrao
Male      1090
Female     972
N           81
Name: count, dtype: int64

In [26]:
classes_yes = annotation_statistics_df_full_final.copy()[annotation_statistics_df_full_final['majority_annotation'] == 'sim']
classes_yes

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class,majority_annotation,agreement_rate
0,1,1,0,0,3,4,1,Carolina é uma menina bem difícil de esquecer.,Female,sim,sim,100.0
1,6,2,0,0,2,4,2,"Inteligente, ela é muito sensual.",Female,sim,sim,100.0
2,51,2,0,0,1,3,2,"Menina bela, menina bela.",Female,sim,sim,100.0
3,798,3,0,0,0,3,3,Ele não é feliz.,Male,sim,sim,100.0
4,1235,1,0,0,1,2,1,"Ela é linda, mas não tem nome.",Female,sim,sim,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
18826,4572971,1,0,1,2,4,2,Sem que a matéria esteja sob a influência do s...,Female,sim,sim,50.0
18827,4572997,2,2,0,0,4,4,"A doença é menos que um extrato de pó de nada,...",Female,sim,sim,50.0
18828,4573057,1,0,0,2,3,1,O homem mentalmente espiritualizado é falante.,Male,sim,sim,100.0
18829,4573086,3,0,1,0,4,4,"O homem atual, na grande maioria, é carente da...",Male,sim,sim,75.0


In [29]:
classes_yes['agreement_rate'].describe()

count    16265.000000
mean        89.315896
std         19.108192
min         33.333333
25%         75.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: agreement_rate, dtype: float64

In [44]:
classes_yes_final = classes_yes[(classes_yes['agreement_rate'] >= 60) & (classes_yes['total_annotations'] >= 2)]
classes_yes_final

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class,majority_annotation,agreement_rate
0,1,1,0,0,3,4,1,Carolina é uma menina bem difícil de esquecer.,Female,sim,sim,100.000000
1,6,2,0,0,2,4,2,"Inteligente, ela é muito sensual.",Female,sim,sim,100.000000
2,51,2,0,0,1,3,2,"Menina bela, menina bela.",Female,sim,sim,100.000000
3,798,3,0,0,0,3,3,Ele não é feliz.,Male,sim,sim,100.000000
4,1235,1,0,0,1,2,1,"Ela é linda, mas não tem nome.",Female,sim,sim,100.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
18819,4571649,2,1,0,1,4,3,"Jesus não era judeu, ele era essênio.",Male,sim,sim,66.666667
18822,4571726,1,0,0,1,2,1,Por isto o homem mente racional.,Male,sim,sim,100.000000
18823,4572368,2,0,0,1,3,2,Abaixo do céu todo homem é igual.,Male,sim,sim,100.000000
18828,4573057,1,0,0,2,3,1,O homem mentalmente espiritualizado é falante.,Male,sim,sim,100.000000


In [34]:
classes_yes_final['agreement_rate'].describe()

count    12147.000000
mean        99.998877
std          0.123727
min         86.363636
25%        100.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: agreement_rate, dtype: float64

In [50]:
classes_yes_final['tem_padrao'].value_counts()

tem_padrao
Female    8945
Male      5073
N            5
Name: count, dtype: int64

In [45]:
final_dataset = pd.concat([classes_yes_final, classes_no_final], ignore_index=True)
final_dataset

,frase_id,sim,nao,nao_sei,not_annotated,total_annotations,valid_annotations,frase,tem_padrao,real_class,majority_annotation,agreement_rate
0,1,1,0,0,3,4,1,Carolina é uma menina bem difícil de esquecer.,Female,sim,sim,100.000000
1,6,2,0,0,2,4,2,"Inteligente, ela é muito sensual.",Female,sim,sim,100.000000
2,51,2,0,0,1,3,2,"Menina bela, menina bela.",Female,sim,sim,100.000000
3,798,3,0,0,0,3,3,Ele não é feliz.,Male,sim,sim,100.000000
4,1235,1,0,0,1,2,1,"Ela é linda, mas não tem nome.",Female,sim,sim,100.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
16161,4570171,0,2,0,2,4,2,Da parte mais alta de um templo ele foi lançado.,Male,sim,nao,100.000000
16162,4570197,0,2,0,1,3,2,Nero ele foi decapitado\n.,Male,sim,nao,100.000000
16163,4571613,1,2,0,0,3,3,O Mestre ensinando que todos os homens são igu...,Male,sim,nao,66.666667
16164,4572851,0,1,0,2,3,1,Embora a inteligência do homem seja grande par...,Male,sim,nao,100.000000


In [47]:
final_dataset.to_csv("data/first_stage_annotations.csv", index=False)

In [48]:
final_dataset[['frase_id','frase','majority_annotation']].to_csv("data/filtered_first_stage_annotations.csv", index=False)